In [1]:
import json
import re
import numpy as np
import pandas as pd
import torch

from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Loading in Qwen Model  

In [2]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

if torch.cuda.is_available():
    device = "cuda"
    model_kwargs = {"dtype": torch.float16}
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    model_kwargs = {"dtype": torch.float16}
else:
    device = "cpu"
    model_kwargs = {"dtype": torch.float32}

print(f"Loading {model_name} on {device}...")
generator = pipeline(
    "text-generation",
    model=model_name,
    device=device,
    **model_kwargs
)
print("Model loaded.")


Loading Qwen/Qwen2.5-1.5B-Instruct on cpu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded.


### Adding in Hard Filter Schema

In [3]:
HARD_FILTER_COLUMNS = [
    "in_unit_laundry",
    "dishwasher",
    "central_air",
    "parking_included",
    "gym_in_building",
    "balcony",
    "pets_allowed",
    "heat_included",
    "water_included"
]

### Using LLM For Hard Filter Extraction

In [4]:
def extract_json_from_text(text):
    """
    Pull the first JSON object out of a model response.
    """
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model output.")
    return json.loads(match.group(0))

def normalize_filter_output(raw_output, valid_keys):
    """
    Make sure the output contains every required key and only 0/1 values.
    Missing keys default to 0.
    """
    normalized = {}
    for key in valid_keys:
        value = raw_output.get(key, 0)

        if isinstance(value, bool):
            normalized[key] = int(value)
        elif isinstance(value, (int, float)):
            normalized[key] = 1 if value >= 1 else 0
        elif isinstance(value, str):
            val = value.strip().lower()
            normalized[key] = 1 if val in {"1", "true", "yes"} else 0
        else:
            normalized[key] = 0

    return normalized

def extract_hard_filters_llm(user_text, generator, max_new_tokens=200):
    """
    Use Qwen to read the user description and return a dict of 0/1 hard filters.
    """
    prompt = f"""
You are an information extraction system for apartment search.

Read the user's apartment description and determine whether each feature is explicitly requested or strongly implied as a hard requirement.

Return ONLY a valid JSON object with exactly these keys:
- in_unit_laundry
- dishwasher
- central_air
- parking_included
- gym_in_building
- balcony
- pets_allowed
- heat_included
- water_included

Rules:
- Return 1 if the feature is requested as something the user wants or needs.
- Return 0 if the feature is not mentioned, unclear, or not requested.
- Do not infer lifestyle preferences unless they clearly map to one of the listed features.
- Do not include explanations.
- Output JSON only.

User description:
\"\"\"{user_text}\"\"\"
"""

    response = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        return_full_text=False
    )

    raw_text = response[0]["generated_text"].strip()
    parsed = extract_json_from_text(raw_text)
    normalized = normalize_filter_output(parsed, HARD_FILTER_COLUMNS)

    return normalized



### Apply Hard Filters

In [5]:
def apply_hard_filters(apartments_df, hard_filters):
    """
    Keep listings that satisfy all requested hard filters.
    Assumes apartment columns are 0/1, bool, or coercible to bool.
    """
    filtered = apartments_df.copy()

    for col, required in hard_filters.items():
        if required == 1:
            filtered = filtered[filtered[col].astype(int) == 1]

    return filtered